# Unsupervised GraphSAGE on PPI Dataset

Inductive Representation on PPI: Inductive unsupervised node embeddings on multi-graph protein interactions. This notebook implements the approach with `SAGEConv` inside a `K3SAGEEncoder` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SAGEConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import PPI
from k3_node.loader import DataLoader

title = "Unsupervised GraphSAGE on PPI Dataset"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Loaders
train_dataset = PPI(root="./data/PPI", split="train")[:2]
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
in_channels = train_dataset.num_features

# 2. SAGE Encoder
class K3SAGEEncoder(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

model = K3SAGEEncoder(in_channels, 128, 64)

# 3. Sample Embedding Extraction
for batch in train_loader:
    x = ops.convert_to_tensor(batch.x, dtype="float32")
    edge_index = ops.convert_to_tensor(batch.edge_index, dtype="int64")
    z = model(x, edge_index)
    print(f"Extracted node representations for batch of {batch.num_graphs} graphs: {z.shape}")
    break

print("\n✓ K3-Node PPI Unsupervised GraphSAGE execution completed successfully!")
